In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import sys
from pathlib import Path, PurePosixPath


def _canonical(document: object) -> bytes:
    return json.dumps(
        document,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
        allow_nan=False,
    ).encode()


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while block := handle.read(1024 * 1024):
            digest.update(block)
    return digest.hexdigest()


def _manifest(path: Path, *, schema: str) -> dict[str, object]:
    document = json.loads(path.read_bytes())
    if not isinstance(document, dict) or document.get("schema_version") != schema:
        raise RuntimeError("private input manifest schema differs")
    claimed = document.pop("manifest_sha256", None)
    if claimed != hashlib.sha256(_canonical(document)).hexdigest():
        raise RuntimeError("private input manifest digest differs")
    return document


DATASET_INPUT_ROOT = Path("/kaggle/input/datasets/dylanmoraes")
SOURCE_INPUT = DATASET_INPUT_ROOT / "apar-sentinel-v5-source3"
WHEELHOUSE_INPUT = DATASET_INPUT_ROOT / "apar-sentinel-v5-wheelhouse-py312-linux-x86-64"
SAFE_INPUT = DATASET_INPUT_ROOT / "apar-sentinel-v5-safe-evidence"
SOURCE_MANIFEST = _manifest(
    SOURCE_INPUT / "source-manifest.json",
    schema="apar-sentinel-v5-source-archive/1",
)
WHEELHOUSE_MANIFEST = _manifest(
    WHEELHOUSE_INPUT / "wheelhouse-manifest.json",
    schema="apar-sentinel-v5-wheelhouse/1",
)
SAFE_MANIFEST = _manifest(
    SAFE_INPUT / "safe-evidence-manifest.json",
    schema="apar-sentinel-v5-kaggle-execution-input/1",
)

SAFE_EVIDENCE = SAFE_INPUT / "safe-evidence.json"
if (
    SOURCE_MANIFEST.get("artifact_name") != "apar-v5-source3.tar.gz"
    or not isinstance(SOURCE_MANIFEST.get("artifact_sha256"), str)
):
    raise RuntimeError("source artifact binding differs")
if (
    SAFE_MANIFEST.get("artifact_name") != SAFE_EVIDENCE.name
    or SAFE_MANIFEST.get("artifact_sha256") != _sha256(SAFE_EVIDENCE)
):
    raise RuntimeError("safe evidence binding differs")

wheel_entries = WHEELHOUSE_MANIFEST.get("wheels")
if not isinstance(wheel_entries, list) or not wheel_entries:
    raise RuntimeError("wheelhouse manifest is empty")
for entry in wheel_entries:
    if not isinstance(entry, dict):
        raise RuntimeError("wheelhouse entry is malformed")
    wheel = WHEELHOUSE_INPUT / str(entry.get("filename"))
    if (
        entry.get("size_bytes") != wheel.stat().st_size
        or entry.get("sha256") != _sha256(wheel)
    ):
        raise RuntimeError("wheelhouse file binding differs")

SOURCE_ROOT = SOURCE_INPUT / "apar-v5-source3" / "apar-v5-source"
source_entries = SOURCE_MANIFEST.get("files")
if not SOURCE_ROOT.is_dir() or not isinstance(source_entries, list) or not source_entries:
    raise RuntimeError("materialized source tree is absent")
expected_source_paths: set[str] = set()
for entry in source_entries:
    if not isinstance(entry, dict) or not isinstance(entry.get("path"), str):
        raise RuntimeError("source manifest entry is malformed")
    source_name = entry["path"]
    relative = PurePosixPath(source_name)
    if (
        not source_name
        or relative.is_absolute()
        or ".." in relative.parts
        or "\\" in source_name
        or relative.as_posix() != source_name
        or source_name in expected_source_paths
    ):
        raise RuntimeError("source manifest path is unsafe")
    source_file = SOURCE_ROOT.joinpath(*relative.parts)
    if (
        source_file.is_symlink()
        or not source_file.is_file()
        or entry.get("size_bytes") != source_file.stat().st_size
        or entry.get("sha256") != _sha256(source_file)
    ):
        raise RuntimeError("materialized source file binding differs")
    expected_source_paths.add(source_name)
actual_source_paths: set[str] = set()
for source_file in SOURCE_ROOT.rglob("*"):
    if source_file.is_symlink() or not (source_file.is_file() or source_file.is_dir()):
        raise RuntimeError("materialized source tree contains an unsafe entry")
    if source_file.is_file():
        actual_source_paths.add(source_file.relative_to(SOURCE_ROOT).as_posix())
if actual_source_paths != expected_source_paths:
    raise RuntimeError("materialized source tree differs")
if not (SOURCE_ROOT / "scripts/run_defense_v5_kaggle_stage.py").is_file():
    raise RuntimeError("closed stage entrypoint is absent")

NOTEBOOK_SOURCE = SOURCE_ROOT / "kaggle/defense_v5/40_label_shuffle.ipynb"
if not NOTEBOOK_SOURCE.is_file():
    raise RuntimeError("approved notebook source is absent")
OS_RELEASE = Path("/etc/os-release")
if not OS_RELEASE.is_file():
    raise RuntimeError("Kaggle OS release binding is absent")
runtime_image_facts = {
    "schema_version": "apar-sentinel-v5-kaggle-runtime-image/1",
    "os_release_sha256": _sha256(OS_RELEASE),
    "python_executable_sha256": _sha256(Path(sys.executable)),
    "python_version": ".".join(str(item) for item in sys.version_info[:3]),
}
os.environ.update(
    {
        "APAR_V5_KAGGLE_IMAGE": "kaggle-cpu-runtime-fingerprint/1",
        "APAR_V5_KAGGLE_IMAGE_SHA256": hashlib.sha256(
            _canonical(runtime_image_facts)
        ).hexdigest(),
        "APAR_V5_DEPENDENCY_MANIFEST_SHA256": hashlib.sha256(
            (WHEELHOUSE_INPUT / "wheelhouse-manifest.json").read_bytes()
        ).hexdigest(),
        "APAR_V5_SOURCE_ARCHIVE_SHA256": str(
            SOURCE_MANIFEST.get("artifact_sha256")
        ),
        "APAR_V5_SOURCE_MANIFEST_PATH": str(
            SOURCE_INPUT / "source-manifest.json"
        ),
        "APAR_V5_NOTEBOOK_SHA256": _sha256(NOTEBOOK_SOURCE),
    }
)


In [ ]:
import subprocess
import sys

install = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "--no-input",
        "--no-index",
        "--find-links",
        str(WHEELHOUSE_INPUT),
        "--force-reinstall",
        "--no-build-isolation",
        "apar==0.1.0",
    ],
    check=False,
    capture_output=True,
    text=True,
)
if install.returncode != 0:
    raise RuntimeError("offline dependency installation failed")


In [ ]:
import json
import shutil
import subprocess
import sys
from pathlib import Path

PREDECESSOR_CHAIN = (
    Path("/kaggle/input/notebooks/dylanmoraes")
    / "apar-sentinel-v5-30-arms"
    / "apar-v5-chain"
)
CHAIN_ROOT = Path("/kaggle/working/apar-v5-chain")
if not PREDECESSOR_CHAIN.is_dir():
    raise RuntimeError("exact predecessor checkpoint chain is absent")
shutil.copytree(PREDECESSOR_CHAIN, CHAIN_ROOT, copy_function=shutil.copy2)

OUTPUT_ROOT = CHAIN_ROOT / "40_label_shuffle"
execution_mode = SAFE_MANIFEST.get("execution_mode")
if execution_mode not in (
    "kaggle_capacity_validation",
    "kaggle_locked_successor",
):
    raise RuntimeError("closed execution mode is absent")
command = [
    sys.executable,
    str(SOURCE_ROOT / "scripts/run_defense_v5_kaggle_stage.py"),
    "--root",
    str(SOURCE_ROOT),
    "--input-root",
    str(CHAIN_ROOT),
    "--output-root",
    str(OUTPUT_ROOT),
    "--safe-evidence",
    str(SAFE_EVIDENCE),
    "--execution-manifest",
    str(SAFE_INPUT / "safe-evidence-manifest.json"),
    "--approved-commit",
    str(SOURCE_MANIFEST.get("approved_commit")),
]
completed = subprocess.run(
    command,
    cwd=SOURCE_ROOT,
    check=False,
    capture_output=True,
    text=True,
)
if completed.returncode != 0:
    raise RuntimeError("closed checkpoint stage failed")
receipt = json.loads(completed.stdout)
expected_receipt_keys = {
    "deterministic_sha256",
    "manifest_sha256",
    "observation_sha256",
    "stage",
}
if set(receipt) != expected_receipt_keys or receipt.get("stage") != "40_label_shuffle":
    raise RuntimeError("redacted stage receipt differs")
print(json.dumps(receipt, sort_keys=True, separators=(",", ":")))
